# ACM TALLIP-26-0521 — Remaining 9 Clean-Split Runs on CPU

**CPU-only full Transformer fine-tuning.** This is **not** a fast substitute for a GPU. The full nine runs may take **many days or longer**, depending on CPU and RAM. No fake metrics, data subset, or reduced epochs are used.

Completed GPU **Fake News + BanglaBERT (seeds 42/52/62)** are intentionally excluded from the nine-run plan. They are **not rerun**. Attach their previous output to combine metrics later if available. The CPU run order prioritizes the shorter Hate Speech experiments first.

## Input requirements

1. Use Kaggle **CPU** (or adapt the same source scripts locally). Attach the **original two datasets**: Multi-Source Bengali Fake News Detection Benchmark and BD-SHS. The notebook rebuilds the *same* cleaned train/validation partitions and retains the original fixed test rows.
2. The previous three completed GPU run folders are **not required** to start these nine, but should be preserved independently. If you have your previous saved `training/` directory, attach that Kaggle notebook output and set `PREVIOUS_RESULTS_DIR` to its full path. To resume a partially trained GPU mBERT epoch on CPU, attach that run's entire folder including `latest_epoch.pt` and `best_checkpoint/`.
3. Click Run All. CPU training writes **atomic optimizer checkpoints** every 200 effective-batch updates and stops gracefully after a configurable wall-time budget (default: **4 hours per notebook execution**). After it stops, *save and retain the output* before another CPU session; do not assume a running notebook's Output tab shows live files.
4. For the next session, attach the previously saved output, set `PREVIOUS_RESULTS_DIR`, and Run All again. It skips completed CPU runs, continues mid-epoch checkpoints, and never starts the three already-completed BanglaBERT/Fake News seeds.

CPU uses `micro_batch=1` and gradient accumulation to keep the **effective train batch size at 16**, with all three epochs, 256/128 subwords, LR 5e-5, warmup 500 and the same two pretrained base models. Actual computational/floating-point results may differ slightly between CPU and GPU.

**Warning:** retaining or merging earlier GPU results requires the real `done.json`/predictions/checkpoints, not just the log's validation scores. Any missing earlier results must be reported as missing.


In [ ]:
# Environment dependencies (torch is preinstalled by Kaggle GPU images).
%pip -q install 'transformers>=4.45,<5' 'scikit-learn>=1.3,<2' 'pandas>=2,<3' 'numpy>=1.24,<3' safetensors
%pip -q install git+https://github.com/csebuetnlp/normalizer.git


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile, shutil, base64
import pandas as pd

ROOT = Path('/kaggle/working/TALLIP_Clean_CPU')
ROOT.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = Path('/kaggle/input')

# Only set explicit ZIP paths when Kaggle auto-discovery cannot find the files.
FAKENEWS_ZIP = ''
HATESPEECH_ZIP = ''
# Example: /kaggle/input/previous-tallip-output/TALLIP_Clean_Retrain/training
PREVIOUS_RESULTS_DIR = ''
CPU_MICRO_BATCH = 1
CPU_EVAL_BATCH = 4
CPU_THREADS = 4
CPU_CHECKPOINT_EVERY_UPDATES = 200
CPU_MAX_WALL_HOURS = 4.0

REMAINING_NINE = [
  f'{task}/{model}/seed_{seed}'
  for task, model in [('hatespeech','banglabert'), ('hatespeech','mbert'), ('fakenews','mbert')]
  for seed in (42, 52, 62)
]
assert INPUT_ROOT.exists(), 'Run on Kaggle with the two original datasets attached.'


In [ ]:
# Source mirror of clean_splits.py (full readable script is in GitHub src/).
import base64, zlib
Path('/kaggle/working/TALLIP_Clean_CPU/src/clean_splits.py').parent.mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/TALLIP_Clean_CPU/src/clean_splits.py').write_bytes(zlib.decompress(base64.b64decode('eNq1W3tz3LYR//8+BUrPNKRzR8lqnMbnXmYU+5xoosoaWXU7VTQcHIm7Q8RX+JB0Ue+7d3cBkCCPks7NNJOxRBJY7C728dsF9OJPB3VZHCxkeiDSW5ZvqnWW/mXkOM4PtYwjxutIVnwRC1aJsprkhcwKWW1YmceyKtmyyBJWreHrXcbgy0qmPGaXx6enJ+cs4hUvRVX6o9ElDFnKexGxdRZHk6yuiB579+kzkyU7m3+eX7Aki+RSishnOPyWxxIoyCxVi7E1L5lZf5TdioJVBZepTFc++znN7lLFYip4MUl4Fa5ZyNMIaYiS8UKwQiQwLVJMw6hYAg0liA8cAh/wP0+ZuOdhNQmLrCwnaulVzQueVkKM2dnHS8aBjyxbMvifxzHLYaBEFeHSjJYWQBF0OBrRWkGwrKu6EEHAZJJnRQWrpFlFwpWjkXlXrHJelMI8h+Wt+RVEX8dyYR5lZn77tcxS83vRzKxTGWaRQP2bV7/LfCljofgJszgWIa1uGHqX1SBfMWaRWPI6riIZVmpwzitc2ww8h8eG5RwUDLuCGxONRvN/nc/fXc7fsxl7cJb8RqTirnSm8EAbBb/99fWbw9dj5sDewtObb757Aw+4a/ppOx4x6z9njXuXCxGuO2S+OTw6+qYh8/rw6LuWDDy92W5Hl/N/XQbvPp72eYFh95Uz7pF2SgHSp6FwtqPT4x/mp4NTY74Q8e5cfGL6cTs6mx9fBJc/Xcw//fTxdEcVh/53h30C8O61nnf248Xx3z/1J73uzzjajj6dn55c4khXa0WrwyjCG41GsJUszYoEHOl3EZBVuyj+lJVV4bHJ9/hzShrH10DMMhy/mek6Zx9+fgeUYTTN9zw/BMdegie7Hk0vBJh3yhzm+L9mMnUd8xNeyWWHbAiCrLJi44be1eE1kylIcI5sf8J/3jlexwL6/4m4FCxky6yAf2VKfHs+OanrGZnLNcSEWJStsGOWTmF4pYQW1RW8u1aS32VFVILoQ4pqKNtCPjRS0tQrOZXsa5Zee8SVRK4gVqyEGws9BFZlKYx55W1RG9b772FdJRMwpV9qIX7lYciLyOXThuExW7QPJMoyznil5AAdQ6ScEXXO/sMWHabV2z/DW3agh6p9yfT6r/xDvTCGiaBc86PX37ro+1Ny+a65RHKF0XtmApOvx6s172S1prDhZzms6xQLx8MYsYZoEYtps8Gor0WchTeoMwnRx415sohAYjXSLwSP3KOXL48OPZDdcbxpxzgUF36dY4h3iVJHaP19Le7Vb67RLUTWAGNbAFHR5UW4lrdCiTlmkLJutM3Q5rfOkkf+e7DgDwVPtBCJSBaQQtAJ0bYPICuEoixFdEBmj6TYbMZaX1a6BkHAGlRm+RqMCdhxWsXpSO3/W+Yf4KfhjzQI3wL82OoBltFcQEJBPZohfgpsxhKl7moNogXw8JnHtZgXRVa4S2d+n0M6gMz4oGhtkdCDXnhruaRhUK1Au6tmEHfLdXelJWoKtAOKw50MQE53CSqGQAuZPl3NnLpaTr6blHIFvh9ndwEQg9Aw+8BBTZ2tJFJ698StjDBWE71GI329jZky3sd3b0AYZ/755P387N0cNvBrIrAjmGaoK5MxLHQHUQbZMsBwUrqlKKQopzj6E/3adSPwmsd8CH0DaeBOaCoNCzDQGD0O8UmfEKZJm47nDQ1cOL+kTkel+HHANdYivAko00H8bH0hWk47KtzxjrMs1XoFkzTZ9wopXBvTjAAxFazJr/2Plo7JRI/BkwoEKcZM/y4BZ6UrEOC3WhZgroBj6gQwDCrrAaltDx6ILWOzwEq0vOpyc+3LMuWu5/N043rIEQzpMdUd8yxjZzWgQEwYB6Q48p5H2ME4P7QeL6tNLlzUpg9h+bdaQD5jf0IkcIiJ8ZWzfZaNf6TCODLgeV5smGLH+Nt0lyu16TpuBfjV3vQyq4uwiYzG73ZdbTdlA8DP6yoAlFyZ6XeA28FnY8HTAKPllC2yLCbbQbCppCv4HaIfbVudOK2YUQFaW59KuSqQgs4VJto2/tP70AmamBBhsSsac02aNvBVbYn+MhQ7dxR/koKwrMjuwCIBRrNElqoA6eob9N9b1o6tO543Zu1II/CIhlNgaNXUDtsx9YTnbg/ZPKG0F+yYAedgzGlNOSVDMEW1SJYkAJFVHfUyS+PNS5bXaVjVVMQclJtkkcXweQlpXNPiq1UBVgUppAFXNNhn5/BeFPC+rRjnny5RgeVYV2msrEF/+EZTGyjaGIgGG8mLlArUWNzLEGiFGWL5CiADoAggp4qrOFabA4UZAckkrzZBo5rIUifgRLesE5cCUyUShevwF1AXzjC794QmcViAta9F9+EpYmiT+Hr7OE2jrABr35jnSLqxHlUFBFACBCp8WLIFGJqwkEDrazi70oXDNcBC6yUWEdfeuE8Yi4o/SJnqkg5p5PbLCBN3T5DdKgeJiiyfUli5UtkLf4N9HTfw+fp6QH2wnlX+QvZGI9JVZv+DXk4Z5yWi4QpQHnYoRCgoTjLqUSwyQBnGbqG+ryTV3W/t/kbbzPCb4AWmH8horCozsAQBLikKk/MbFXudqIYmS0VPFyeDNpoJV4owZJwocp2+G5DeraCEKFYzsKPy/RfBsBNoI3qWOG3xl9Nu1el4eytRGfveKqTh/18FDi3RU58KMGBSAZiarl47mNg8jNnSURkooOFtO4xKDoMGNZhCqEHOqBZH0KHNNqBwZL+QaSTu8YUutQNdrTrbkaVGQxrAVFkvEPnYXD8Hq5wzbOQ1PFOrsIQkmXBIlVhgR1qCEsiJIJJlCLGepyFgZRDm6rqxApWLcQes5X0sOKs6xz4BiaOrjpYr47QzanwQEb+jkXaXszhCrZClgQYlahPTSHcSLQMRpf1i6bsl9oLan9TpZCiTgOIYZ8IekQt8DTaCVQlnJSkfEx52VVKOyA9LvrcWLRycZ6WKOtiqLQToBPJrxTEtVtS9/Xhx8uPJ2fGpWqBFMr5tx40+ZjomdszXqIBNZg24aoLUEBmD+8HsEYQ38/X7toeClPQkzLkw1ih5cKhJCN5e0O0HHpFZWIaG0oOSQFHgQ1C8PdBe2VCNLA622PRomoaTzaov4yy8MmLtQLMxszp+6p03frT31VlBS6gW0LrYi35nH6h3pO2w58Ye+xvr9jF1rTZhr8TkzV6KPSf7NaFIqwywLSgXoVuUhTUiSqzi6ipbLof0DFzyBVTRNHfyJMce+x55+7bL2wum2KBTDEzNTYdGRWVqAXD2AxhQzNsWYPEWvEKWPVLohLmAfwDhq2JkgjEJsCzxl/AN4Iwl2I3P5vdhXEeCfIscuUeqNTYoMO9gooVUcY6i2Ia0DYPKLpahrOKN36E1EPx8niObbjecT43X7MT1qXHJ8ZOt105j3lAACwQM1yzRxMH9KZF6mvwxfXKTv4As+AnmwkDvzs4C9NbbtrZG6df4buu2KgXf4PGSgqrdTGrS8Qv2jo6LqIjTsRtsDQIvmBolcPgXbKKkKMxXEBbLyraoeAP7v6hlXLGT97pGwXJsKe7aKklR5LEuwyD8vWRk01Ed0gEUZAW1mHV4pUmZY6m2+AJDw6OgFJ8AON2KeDNmCyxkG0NM2DpLQWpjdWqz+qn2YaDk7h1LWMEYVLsHYDHK0w0DXUEHrRItCEOhAkZmMRAGCnqstkp7YXhGhhGKYBKWVs8evl4pIla60tCLJj1INRgrVF7S0ZnKXrqCQ0CRlK5HUSvduPc+pdgSY4wBcY5a8J6ylqLibTvJsVlwYvjdM9qCDrO6bANuIinMdU0sEpXqDykr0TYxHe5WtRuO9T6qDaxCRK7R46Tl1uv09JXiUcZ2elcKkFQPGkzhpr6e7vj73uK3CRzIU22AK/YlPXhQbGx7x07WtnaxuBKosUIA/lXQUbHqsexosOMypuS3C1ArFrbUsf2ha2Ct9F4IdKy11crDs9uNGg+tiBuk3aV8VKSptZstFV1w34i8aioIR7WkDEwJs3zjel8UI5Bcq6XWJHzcGA3XtTVaW+V5PsY2xf49fZldFrU+SVCBxLTODKO6mCZOv4jFllzLqPX0xzl9tnu0y8HzPST8sJRFSZ0QiP3kpW5fyrEtt6o68VXnfLlbNbcsXxF57NFY79Ra+zVTTwW/4Su6rYFZUrUU8VQSfnSa2LTQdvKgiG9NKlbLLimDg/6sTjQ7YA59hbSCHWgHXiAt1RA0CemxmWaAMzTcT24iWbg55OW0KmkvxxAeoRYLshtra0FVO33wnkl9GcFne909d/KrjFJuR0sHbdbEtOqvfnfGzCqKn8VdYZbk1OvN0tmDk4hqnSHUcla/yxxNJqlkIvCqw1ZvUiRCqbBQp1Lfw++6CWbAtXpdFbOQhYphyxEOU4vdMRiYfsJz02WFMiALFhwxSINvdT9p9Bz4VKkdNfAffVdgl1GTWDyDQqkYcft2eNBiIn1/yURqhYD0weUdqCoVd7FMxcxxds5YnZ1zTDJDNHPc8PcyrP5JL+iAdilFHOH5cTm7UupqFPW0hsat7BaKUkv59GMtOEjlDn/EjOU2O6ZNRREMVPPeCojUidXXllw1SBnHMDq7xQNvhGftGBuFPXLt5Emj3Gr+CC13u8mWibVvVdGIh1kBKheNjt7QUf3wOHUmjFWLdStDfbcb6Y3REG2sUFofVMXIJ5n8xMs1W0DlgjV3U4CqDtNbAmLifs3rEksBxyJO9HT9FFRr8HO8wwekh1oE/Xltz1GVTzmHqG2atQKpIBCnLr/dG/T6dHBe0C92N8oqNMgZKIX3IwMDcS9231qzG6PXqMpYYf8ccSjzWmRU2B2gYYXo/YgIunsQrDpWYija1vJcpPf2KqqHWBrKZuqGC14GsBiORAyFh4oSyo970tvx+xmfGyJrRwiM0vbz0B4qOKyP0XDLK5nCC2wJtVT6x222u/Wb/rihwWLTNlR6A6y5GuMbh23hOpXs1D9ooPZjxgiVcaV1rkShuWp1wy5mW2uK6mCQD5AfNn4ZmEu0ASzYHqIokRQGg489anSVN2g0RBRJe+o2SWuQ/QsxNtz2dggSnYYqHYXT7hRyUdPJDlpNVbg3ngobt8pYII3fqiKyKTp2L1hQ/Ne763qmWN/1zUdW1SD8idUtD9579f36Wc+EgxKsI63kUoZBLBNJxnvVLfPmVO6DUIhFqhSQGrsDPACWQheuWU33avCa54G52XlgHe9P1PH+BE/WiyxWR/Q89p1eNaluYVM7PaqxX4kJhiI+qzJ14ftO9C9iq/ykDh6wDWqdi6IN7iyChyO9JcwpCRQLeV5k9zKBtwcpMN5mtLesTrMFdb6izkVtatwqQx9cq+m+Ef/qfoYRo4JJInrbeas6GnbXhbrPKC41xEWxoSptZ7EzcQefwVCU8JDBCxmW5hQNlCUof5dikmQQ+prb7wwGY7ua6bzQfoDay2/7X9fWkTUoHSzYIMs22M0eFJy5+qqT8L663o4V+XZAm8roq2MJs3QwoOciakfvpgCa1STlcvawAwK2EPWXEB3XVr2jL68psvr6Em6d27uERjfpEd6aW/X+cbGiE4dz+gIwE6xG5qjqWRBEWRgEnjUT2z0B11NcZzIxdzgnqqTB21ozdbHJHHNaXD5Co73E/UeomNzxR2hAbTuB+vJ/nZ5mE0r8EzKCCRXTY8bpTwpmTlkhkqqAiAYXMBWRuyZGP5Bc6baffeAIou7ede8jV2WBkI2Fqepp7q1ZdzjLOibIri677dx/M1ff2ktv4w6Xex5EYCigaWkW9IHSEOzCAGEvr5szzfVhzYR5Jgn3PhNx7b8h0JTaN0RLJ6SEp3Kp7ndbNQ3eLSwkWgBWFepPfCZH304OXx+9wltY2LQxGSLGK4htpAKn1SHJrivKCmYgLUppwfnF/Pz44vjy5ONZ8Hl+cfLhZP4+CH48/0dweXF8cnZy9mNwPj97Dz+dYeBg/jYISJLZDAA/fdnI3B/YHaghYWND+g70cB22a27jbgmowCja2tiKvRXME5UKTY1FdTpMTaNKQzqzIT7+uY9jEVFlM0EvF7/5UZ3kpWvGqyZOWs2OsCFQ4l8gcUALUl91wMvnv6RDzYJOjvhsUEJjF7hqBQkXkvqDYqQfqkcjqA0CKnODgG4MBAGG6SDQ1wZUzB79F/5UnY8=')))
print('Extracted', '/kaggle/working/TALLIP_Clean_CPU/src/clean_splits.py')


In [ ]:
# Source mirror of train_eval.py (full readable script is in GitHub src/).
import base64, zlib
Path('/kaggle/working/TALLIP_Clean_CPU/src/train_eval.py').parent.mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/TALLIP_Clean_CPU/src/train_eval.py').write_bytes(zlib.decompress(base64.b64decode('eNqlXHtz3DaS/38+BZapK5P2iHo49m0mN6lS7HHiWlvW6ZHdq7EOxSExM1xxSC5B6mGdvvt1Nx4EOdQjXlUi8QE0gEY/ft1o+oe/7Day2l2k+a7Ir1h5W6+L/PXI87wTsVNXUZqzRVGv2SKSgp1VUS6XRbURFVtGmzRLhWRFzv6YnXz88HH2nsWZiHKRMFlmaS3D0eh9AS3yombppiyqmp0VVbxmTV6nGYtYJaKM/XZ8zqomZ6mE+381QtYiCdkJPLlOYeCdnTKL8p0iz25HdcGuoixNolqwNC+bWu7GRb5MV00V1SlMBP57B+SwY9HUMJ0o3aT5im2KRGRAXjYZTgtWNxotq2LDOF82dVMJzs0MoxymS9TkaGSeVasyqqQw97G8MpfrSK6zdGFu/ymL3FxvonptrgtproCFSbGxd5amXDfAFHPX5GkMc4aVRmqiuOY63QgzTXM/Zvj7W5EL1a6EQWE+ptkxzsEQzZtNecsi2JDSPCphNvAA/isTRUBewh5WebgRdZXG0rIljoHJ8S2XcVHBqMj3RgKXOCyzSm/GrKxEnNITuIiyjC+pKZdNiRRGo89f3s8+8Y/vT9mU3XmLKF9l0UJUtTdhXizFohF1npW7zosx8zamBf7dQSnc2cAmgujlqybKdmJ4knj3o7PZP874uy+fiPYyuhS5uJbYrxY3RGgNDJOlEPEan0qR1yKPBfT8dPgrTGuoK84i2+6Ld0zf3o8+H/6Df5od/Xb2e7//wZu3/b77B3+9H53OZsQD/8eDMXsD/789CEaz4y/vfsenr0dnJ4cfj/ivh2fvkOT+29Hsj8NP9v71wejT7PDk6OPRb/zk8GwGj96InTejvx+efD4/5qdns2Mk82ZvbzQaJWLJEth1jnLpo2xMSCTGrFj8c4K/RFwHbOcXdgQSNBkx+MFWIYg7cCjcXCZp5asbOT2rGth5cZPKmheXdBtQl1psShiTeqLu8TzaCBouxCv2inlhvSm9tnV4XaW14Lg5Ps4txFlKHyY0Bs1OYLgpMEbkEnUzknGaTj9EmRQB0vqae/gOFASEYOo19XLnry7tSoDJiNUEAs2FZZqBKK4j2BSHD7R0WVdq5fAaVqE1OtSNFV0yRbSgohS571ULL0CtWaqe+AN2kcXrJr+EBTBYXOVn0WaRRBO2hBlFiX/w8uXBXjBmC88L2m564LApUaV9oqDGrARYppxersVNkq7ANPpmPaBtpWIf/prgIoAyqQ7PwUKDmfwmusv7gX2O6ngNBrleCyRwlRaNZPAnQVVAMy0WRXH5QhL1qoiFlGg9UbFLEWoG39TApEqEsln4lbeu6/Lr6av/u76+/hrCBaoLg18wKs0sCAa7/df8f3+5ePWLaU0t3YaO/Qvtcnzv6MO7odaWrB8GX/fvXo/voVXlfd3/uv9o86/yVWcGIcw6Lf0u+2Ehfb7qldnmaku0xfRrUhNgYZLG4MtoE/BSq5cxk2Om7OSYLffHjKP+PG5BfSszvRHGjCyVnM73xmz/YsyiK1FFKzH1NlFcFbC6b6IqeJJeEfXpXmeBd56x7WCgllkR1X7X2G8tKBh3pFeNwu3sLRn7JBibRmphtoW6Haa33LfNlvv9Jn33wxUH+B7HXv23/uMMC8K6yFJUrnu9laivnMAMV1DGVzdgDY0BrSN5qdVuE+XpEpRzQtts93tOL8skfA9S/KECQ3ihRGCJ1xLdxf3IWA4aBi2H7xHqQrkErIN/wH/UrsVAUwad7YzYLk0G/viKCppbgCjh6pu2iviTLgmHYeeQTLj0e1YIhgWM9wEaHBX1h6LJk1lVFZW/9GY3pUC+WXyHVCbsDv/cO2MI3Y4rS2r4MvdwftK7mOPfi3Alat/TpDhMlK++paU2zrDgu/utWXcIA7dca47XAfvLtNOIxiBuDC7yUEqAEyAgZoXv9MLenf7BTn8/BLqwhculqMDEIyqKmiRFBrRbfYdrud+9o0FcJtDuojYnoRIjeUVzBKErrvlGbIrqVjszd5kZuBXqSmt5gHVzxTZeFdf4jMa+eNYCP2uC7EraXcTFxrDPAFVTuUHf8MiyYIo0vblBWmpGF2Eq88gPwii/9QPaG2plUdVQs2fN+KjJMmWxgSjpK6qHmfuD81SaDfyXYDJF4j8wnUjWt6Xw0xysPjgbiDv8oLsfig7shTISz5rzeW6E0PS/U3/vnzV3ZRj0tsIS6N611aqB8TcANTkGR37XACnbJCcMbRoaIfAIFAB1H0khEvME2HBBdovukIheL5KHicwvrKEiUwOLUWN00I+KsuCdHq3DMTJyMCS+V0N3XpvBwqgEgJX4d1tvye7jqGDg8Q/6FByHXwr0XHQNzzBE4HQDD23IMacnF+NhqjgfaI1/yFOBOxH5qkbE3sJ7JTnwXpRFvEaArzD7AzTJivMFahWXgBmgvQPskQyY9+77Fuc/QJMiM0BkHMJd7NAJBIDkdVRtGrCktShxfm5I8ABFiJCl5NciXa1r4gEZpkeXRLZn8kwLpZ3ZQ4xHFvw5eugTH6KGjvLPTg996wC9+w5EQsHUSoe71iBQ1/IG+CQR4OYTgNExRuOY4DD4wtUuJe7UK0QiGmUiJpEITYpFtICotk4JGcxB0LTSUehBZMM0B4+EWJ0E3LWjqF0kS6heak5d/dL4SLWZqrZhWZS+inIl+F0MAYNOp6haEUwBoHMFGMlXiwxoNNC/KxxMUQLnuJGInzpjFiuATggEcNX+y5dILwjV405LXD60U6uUxbIGHfRVu1BBQICQSbqZ7uwHYVw24EgomeEHW2QsFwHl1GhJ8KEFeP32wHzTLi9DmB8OjD0ARANIovGG+1oXoXmaqjxXG0e7P2gX3dZauDAdhezRgYNqoXCq7AigajoelBYTEIoKM3O8anJfJcWMN9iCrkVTu7dGVbgTF6sl6NwPbYtGroCF1C5htkqGGKCZZghzP2l1wGspaqdTmzq0OaXDpi4+o2h8KKpTTP2BZL9Dg5Qu05hScGNqc1ZcihxMJNAFWMezNAc7CLHJWiQN4EDKOCjT145nY7XKjGafYNy+yNtobjRykKZaXNwkESAWHl1FKexK1tE15ftPMIm5Ecrze5jEjBaYVZrAZmHwAXFzXQBvq8tO2hQ2FC0ivoZpxMWmzARaXrXdZitAKCgrgpos/c4GKVBJkf9WCqRLhKuUAeIIBy53iOmwmLypdaYKGmCEQWKkLCgYSXvful3nIflQZbFABCkumRpJayMU2xWul9SH3+Hve09tHXBGrmlwQ2SXeQnoVIj88MxOmXYDkUyRJV3u2bZPMk4TBwoqRLGs0qFJB5kb7oIptB0UL1Q7rYMDqLErOTNcAYqDlQXKhq9BOlQEIhCf60F3jWZ7EORhZkJzyVlAWYG7ASh6+rePx+xKVKBLFLBpxkIHAPm6G6ZJllkj104uz7E6sC53P5+bDWwTjs4eqomrXTRCE4y64fAjUffYCKlmRJvPIQdFzSds3mbFbsYdHVc+6wYd1nAUY4z8Rdd229B8rOM6Q0B2XZ6NOsxUnhF2mCGfNQiNQmiNncH6tH319V9H/tEXcA7bXXPuS5EtVW5LcgNKeD/5CG1Cw0nVdPu9XZ+mMeqNB3hZD9cP7pQoYXTrEAr6/UF9cKl2ypgDvhkm1c53Tq0olLGU9bORznieoiVbCOCwoKynClHWIGjosJFLKclHor2USoySiYZnvx6ezlTaU53ahGiqfPyl5A8gw/AL5UVAWBsAuIPvyMM4DXiUZVuNFlEMng8gCrTO8zAB61BtUJPrNIa9QJV7uPUCrNya/M9UofqRbqudKTzuONcQOcDb1fvGsjsRFfC6gZsliLGj8Iqp02f4860x7AYPDwYgT2f28DggXeWYDjXJCsz0wMRlfyaIVD3kr6fFDEUsLrIMYTsplCNYmMLNV4CpblG6v6Wl/1I1aQUUIc605ZtPWqv7BWiSTLg4tQ6yjSAfiFVMGjdXfNHGtIwSckvqTsk7mLNcFpWcemXtdWc1N9j9wmJn1di/hQUlaGmm6nFW5Kst806HiyNDDKyIa79cI6OUTWUljBXRt0E3eUktlUXE0E4FIkC1hYa+GcuEcqwNgadPBcDteUmzXGZCZdHGZm/5Mp/qSyU510V1CXDT5LspPHx8Sjoc/J45fe+kKKZWwkIpU0B4sUgzHy2mMzEVRgfgSp0sgiJRlAAntEar7aYn4WESbf6u4lQ8zwN/AuZDYiSVVdNO6kDRMYAa6TyBs3075hNMGfghJjiJiunTaYqnyBlIrQl2ePpSp2nUGmEt8WVZgP/tQkzcH5AOSu2EoGjUeKFgXtVtSk9bOqrpGmxCUd1ynZt3mtu56SYOkNVP2vwakQa1gCc7++Fe+4zmBU/VI1lHVf8Z4NZ2Tg/CYmMNIrDutjWasBIUQ5siD+JrgGgqJyQ5Fl4MpKuB3tzg3YsW72o4PN/CzhcD2PlpcHzihFIYAwM2Uul4BxtjxrWCLrlrHpXU40JBJlANMRT21cwoBKGnoFAtg4xED/eyr7d7Wr0Z7mlfb/dsBYBa6luvxaBGIMZdMaDW5h3aq/YBNXEodGWF2uVgpLfaWRxT0yzV1FU1EK9Is3pzd7FPzZ1OlJgZ7qPzO9DevlZ96IXbRyV5+j0JNHW6E2xSsgjv2ufSHdaERCez0/PPs/aMwwaj97tOJMoikCni153DvFf7/WDJZsRVY/B+wI+V8J1OY2N7Wkn/gZ3g0XrSxMJ4DNAZ8EIo5YhdMfsNspzdsmLZHtK33opl6RUEBpbgSuRWsX8TuagiuPaDLfAJAaOa6Eu2vwc/rx3mKnv5uGfUeVzXNbr57GfYbeMgFbpZmblO4erZ7tIBeTgh35WQmhCHRBSzp42n2aI0uRm3+VEBhHFsPNluV96LNkym1CY/NaR8bvKztSd0+L6qItyHmtcFRKW56EXc1KOpy6Z20qVEPeilVGl9qmmId1uvKQ4AB5v0kqPthNBL9l62NmzgpcPaV1N9JE8jQUASxWs/6Gur9hE+sB3Ebj9g/4H1SGwKGzNw6qO18wGt1EpJsntnVBF0GGjDxe6d6+zv1YYIOaSqBpTqbOuYcVV10c/ot8B1zNSe97L6GrlumXF7dKVOiLyJVjjgAJ4qYSbDyJvEIwqHrbsdHPiwMnm6/FFVWTir6R6ZOy/mbTXFBfvFOpSerLe444Gu282NS7Fr3G7yJ3JGXd2W0ZVwo0RDDSL9aCnAplUUtbehU19eTUj7ECEnjrLJKhfCjc2eBq4jrRHa3W0XogAEmmj8Q8eHXeQz6cMeaOO4YFdKemUwDlCZGM604AKLavqoZNLRcqdpl3Ifkkw6+t8ZwWCSieFIj5RFIRMHrLhQZOKIS6/vFryYGBiyMjAEp9AHFBMHe6ws9uivsQ8pJtpHrjroI9A67iKHiYs2Vltowxmotfg1FUA6CLwtg+zGFtAw7ATyGhOBoPo0xhhpObBU2nJGeD52htjGNwQ2mDWSu3cKfdyTpWCkzOzD/vTO1fEXRsdfXEzCt8v7n2m7pnfWUODDIXP6A5tpw0lZs7XIkh2suT6bnZ4xDBrgFx6IRkuIOllrt6iGN9E5IykyEVNiyD2DscZjO4wZChJ+pbIWO4Am7UR6qcQCF4wjNN9pgH8ja2XNiJtqsikG62B0vRmWko/NEaf1Nu0cxm5m4kGfo1ITgc2tLDpD9X2ZS/3POTRn0nRE1I1ml+mNwEMUPD9qG2K1mefUyfZI6JrZaxg3F9eYVph6QxW8vYJaKhFGQIq1bOrGXwa91/pFce1DjFalqzSH1ZpCAJ4mWEBXNouMTj841gFyW9Zt6wHdR8WC79mrfdfNE5Acs+1iwjLookrMIna3bUAWgh7e3FoODLU1EqCkcr4HmvqTUkq83de3F1ox0UXqnCEe9Jscxq52KKHz3uvlRDjWPvfOC53mASqoc2/1k8FSBfOOvpzxD1/Oj9572ykJTXrqtnpcrW3RIS6Bbc1dVTNilaIXdI/TWw/toUVtqJb/t+NzTvEK5rzeffl8/Gl2NnvPD4/ec7RYHNN854fwyGvNu2dOzXhTx0DEfHMR5rA/5rOLEN4FYSoLPOCNOr72u8DBqOtaB5xo63S7ULBjEdxFoMp2dANd3HDjXnKLOwxvp9zbVaf7I2jB07YJq3JlDBYBd+UUT7vM1BjNlFlt3VFFfzjTn9FwmPq5HZVuVt+CkBE3OTa9d/e9Y0izjWMtI4F2YIedY1EYQiRSuS4sWyGxc7NkVB4Ari6LqpVwsq4yj0q5LurwWYk452WTgy289AcqPXRNR7RaVWJFWZJuyQYWJE2cYr1ejfmqKpqyV2BMn1ipUqZ2MngQP2V4Ttue8tONc8TvGMD2dL492Hf6mjuns3PMj69e4OWLi3tv4Fj/6aP9jpZ/9wm/CnfxQHCqqc0HdaQb8yiWYvIJNiaCTj5VSswvgjbs02WExANVDjFmL1/SWDo+k81mE1HC7w6L9UhmQQ2+oMy15//6izipinN/NuBFlU0yCKUY2FAsa2nwUzl0Vmp2QGp+0e6479R1AM6lLAVVX6rCWL0inbxw2Ax6SiVo+KYTPpIDhIf6A5eNdNKHAlaZ0LoGSjSd8ky7SspM0ZRTwy01QEt9KA728BvBtHL603EcfVNFwQI+AbShDvEGKSjTp5fFMS63HxpAYIH3PrIgeE5vGeFyuExcErJOiMKYJUmxnO4HgSnwJrrslyk7UC4Tq8UGR0lz/FojwQweSgJQpwXd9yvG1TLRr77u6wluyLw33Z/elHHNa/7uI50ZzreGHmAD22E/hq/3Dt6+OfjPg59+Yi/Z0BKxtgehnvxXVfuvh3g3RPrV95B2kttKneZG/i+MLqrV98tRWqPlWcvKW4FEy28KVDTl7pdA6pmtvsb0o/lkrpKEVM2HouFhtWrwSOKY3viJkHGVlpSv4DwpYs4Dp2cYJQmPdBff29khT7cDc0Wojse4yuwb2XdyHg8QMKjie/sDq/6d4VXNLbhMrPiceviVVbwu0ljIqbY+pv4G6zqVQZ2SItvnjy+PCswfHcCWfvdHaF88OoQyL50RiBUEBToEle15lJj9hBmoRLE6ZkN8JCgi0QBWV9tqEvQHiUgNEEDj8eMJyuXTF9U5U1+SupX12J6KUdwaB01RKFx9DnZHxecGd+FRgZJP7WJ+PNh9c7D79uDRqkQayrx62vWmS6cmG3D5SqgzQe/0+NPHM358Mjs+PDk8+/jliJvPyTnvoPbj2dF7+PtE2HCeo/KvMP+X7DY5wS3z5ZIuVLDK0WHsgxXjZYGYmCrRxA3sHgdjKqUqUePFlaiyqFRr2Rv1SsfMxxLEKyL7RC6DvkPSANd+mIl2ii0jCMpMrDNM2yG9VUlHrZ4op9Mfe2x/VzJ2BtHXSgPHrsBZIQ61of3zBYNubzTTmHmgr0K0Wb5zIrrjT4dH/MvRp//xaAexJzYlM64DUhSfk/MjfjL773MI8jrB3TN/PFk0FRbcbwVubpDc0QWEIdpp44TuTV5FJen+0P+AQdLqH36HFeXDMkpbfYeuHmkF9wQMw4G6zS0eTHr5QWIYQ4ZNwH7YmGlMNxRim2QSPNBeEdNp9I8ohI9Wio4652w60mjzITilMdsPJttFqmeHJ2d0lnO/6yxSVajOX6DAQbSwq+5sWIGPBmKK4Rm6dfHwv5ZYRxNcmTPC3VELEmsbiXWb03yd/fUOs6z9ZywUirfhCXuHxojZuAfnRtzCzX8UkfTZPxqlWDWJ+WXOKavCOQISzrV1VOhk9P9085Tc')))
print('Extracted', '/kaggle/working/TALLIP_Clean_CPU/src/train_eval.py')


In [ ]:
# Source mirror of cpu_train_eval.py (full readable script is in GitHub src/).
import base64, zlib
Path('/kaggle/working/TALLIP_Clean_CPU/src/cpu_train_eval.py').parent.mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/TALLIP_Clean_CPU/src/cpu_train_eval.py').write_bytes(zlib.decompress(base64.b64decode('eNq1W3uT2zaS/1+fAkvX1lK2xJlJsq6LvEqVdiw7U5mMdfPI3p5qCkWRkMQMRWoJch6Zne9+3Y0HQVKS7U0u5dgUCDQaQPevH2i++tNRJYujRZIdieyebZ/KdZ592/M873R2w5ZVmg6XSSaGZZUl2Yot84KVa8EKsQkTaoG/BLuenJ+fzViUijAbym2alEw8bkWRbERWyqDXu6wyyfKMhQzJ+llewqNMNlUaliJmH2c3/YBNl0sRlcm9YIuwjNbw/jczk2Qnb9l9EvZWRRgnQJSFUVTR8CTPAvYBeYR38E/MorWI7rZ5AlMzYGID3BRCVhtkd5PEQ7HNo3XQu14nksGfKN9sq5IIhWn6RJxnEth4x4oqo0XiknEv1Go34RMrwzt8yJ5YHD7BCmHDer1lkW8Y58uqrArBOUs227wAVjNYL9GXvZ5pK1bbsJDC/I7kvXn8VeaZed6E5do859I8FWEW5xvzq4RdVlPHsJn4y0xsfg+oz295pvttgWqaLEy3GU5iqGXVZvvEQsmyrV5QidvKxX2YmgH+dPbp9MerATufTi4vzi4+8svJ9XTAri8nZxf875Pr0x8H7B+Ty59vZvzqejq7GvTYnv/OJ3+fnvPTT+cwevo/1+opXK0KsQLmBywGdjjuyIAhBxU27iW2TFLB5Tr85q9vB7B1d4Jv0xBGbgux5aV4LAcgCGHMSU45yans93qwgikbs7nvLWFMJh6kN2DeZiGKEh6kEHGfBB+fWJIx/7tvBuyv8P/bb/q37A2bE0N+Gco7mDaPRapHUTupDLyikd4aFiC3QkRrb1C/p0GqwyLMVmmo59ZM1B13sdC77fV6sViyvIhFwZMsTiIh/VRkq3I9gt6lYkc/kvTTc58Nf4BjDrI4LIrwaUSzgCBfleEiFaipR6CYQyAoQCNi1DqSBdQImmvAHpJyjQqTJpHRMsHy5VKKMkCVQIqFAHXIcCIltwHwGlZpyYts5dOC3iim2Gt2cgz/fdsPUG21TuqF9PUiUVuhHRjkoH7JUshS+nkKq4Nlw/Lg+NQjrW6R56ld18xgwXAbRncw7/+ezRCVQOeTJQPdVwAGL0BxAKmufmHrUK6FBIEshGgvCAb4VhSBg7mHxyy92zn+ezv3NDUOus1XvyVbLZreLRuPkc8vH2CngQ3cOdUyeYRxeZGsEgAxkHVZksDzOFnhc4fQHg6+mk5Tul39cSRdya85wXuwC8sEp0ljXpsJPEQ9zYjJskANVufLEa9GBFP7dR//Az2/T/JKWslQg9i/2QVgX0sgoqooYFo9Jei/gx5+Y2rFPohIzSHuX5PAyHKmxeO6qIQZiPauwxzo0O4XgXhMUKr7HZofwlRaoi6/HRp99qexw+8BStAJFo8QG6R5GMsurYBAE8XAF1mUx6D/Y68ql8P/8vpqb+DMmzQa+/f58ZqpfbpNWm0BYFtxsMt+lGfLZDVQOgvLLLSEsLwq3Z+flaLDUgMEkqjIObkkBkLBErkNDXK19wE2UxRPeswDwAWPYSNScCFGbAn7tAOiLmkjpDoe9rAWGRjrSgIkhegxxVVB0FzP8Y7kTPWMgV0LUsY5yItorcwH2XL8GVRlksqAQE53ew/P53ByiOn4DPjtDALclqDoG1FIM2BSlfnPaLU+5MWV+FcFBytO01BKUO2IcHtAfa5zQATw44DuSpQcFx8WXMIC4gqOGe0HfwgB7Lf1fBlMFaY4yMxmW9AxWWTc/u7RKMfy8jvxpMwdCKQSEQVy3u3A/rY93UYco4EtC8GKASB4zzjy5ejZDng5wm78Gf9+0dagIumD/lrw2JFCxKOaI3heevtGBps7+NsHdxCd5TGeJ4gYQgDP7+inUhKLG79H074U7VCWoJdZ3BHzSLpwYs8AELV0saoDKNTvs5zVGBuApPieOhePgEw9G8DcZ0A6tsHSsgvU9gvMk9dvQcOgCwTOoui8wgSUEkIZ9KanRZEX/tKb4vrRI7L7A4AhNwgO4DiUGGPULI7YM8rWi1cveVuAFgOdq58ggApT3KYnAsJUYGSk+w8AMCq5dqShbWnMoTRwhvwF5DbY5LBzeZZEfp/9MG6h0X7jsCxgeomC0PabfYu7A6N/ZteUTsJR48BnkGtY99z64P7joKHCyrd+RAeCZpubOED5JLdBmaewxX7/tnm+OAppD9QwO14GSSk2IJEv1B996dRhRM1hww4zSSjLp63w0Su2E37pFOi7q3Z1ChHiILuGtWoo9fW/jjyhJePgqycl5+AEp8sBUwHKU0vo8F2gXtHjEyxF9+y1qIGjrIm1BVcdK7z3FY1+eyhoCS7J8pLspmCZmSe3hh94VNS0d48A59fhT+32t14oUwRCU4Ex7bzTVgMW27AiARoIDrJUqljfN+i9gP3lhLeI6WAx+RKO1EVPCrHGX2C4OnPYndg9GUTMXEnZGAKyZAWSDW8NBsQc0xgK1lG0ANi2lefwFNA0vj4SFeKkmBbxScicg5ASDh1FGUID/7V6a1+qlMm43jqfhFhKwrlHrqKosTWBtgmWcMCpLsECqG3RhmkbxgTe6peSC9DrTOaFHHvb0muxNPfU3mDUo89c9fafYDkxat1YNad5tupgG9HQW+NosatchDRzbIfDUJOpX7UCo+ZSN6Ww6L6l5O0Axdr18WM59+AdHirNS0c3rr09EPl1tVymYkwIeTgY0afIl9lYPypReciLO3CkxscGJ8H+7GMGX/4x3HwpO7IUW8nBXnEVko8pBRVEIkkxEFdcocB6sL1HbsZHjc+3YG605qpjpZZgEoebf/hK4OFEACxLmNUH4UyLcSONpPnQHiLS+Yzj6AThZnK9NHrNaUnj3eko7GZyGrpjewdeM5XtUnzVnnfTO8INhYNUqUXQAsZeOeGM8igpWfJxdgOLK5KtcrAXOEq5jw41aq2nUh4X0cZQCMLxbS4T5WQv7LT6eXkCrnaaLxBVYSlA+Hig/wxPAnwgauCYwPmAr4xO1ApiKlnCEjANBvKmHADjUtSM7PD2zDCjwyFgsu2P0LMF6dYIQtAHoZBIVutScsy8KNlteIBE8Q/zAR1q/49e4KXKfGFIdaQZjdZhtgIPDjGo4/UpRcDdUvvOMWPl06MJT+gHaFlXuveNsx26Y6067RtrO3THakmBM9Z9dYOTBbKCV0tj3Z3aNKy6TdTNIQJH7zWE20PYphHNE9ilCPV0mqydq0my6UQ29UQPcBpb3duq4siWSZmlT1xlQWNObsSKJL4l5GChWkt6xbQEhVb+UjcMQNRQ2wpLPZ3dBOx6Lag1LvItBJ5tahcfMYcmwg2L6PoBTgZMagQtSE4ReQfeBviqhaD7DYryaZOCr9htarZbfnxgdw2atvD14AbbHFrDvyzptZFcdWWFyeSu6LrOZ8kbw+ieY/coBWU4wr42o+iVOyoASPP73XDucnp18/P0vdZ8zN7QasfP9M+bk5cBgQVXufLxc2N72/Ge9QxleC8cu+ATMcz4i0d9QoqeA1me503KfJNEO5JH4K1mUVrFlNCvbWdtezF8BFGySSXtBJeCzN9z4+wMWI+Yycx1AHek0w9NV0Vr7Ig1VtPS25G7vNZ4V2FHrtQhCw6WjjTsOujXb5FqQ+jIQd0Dw9roOXIA98AwA6Sj2hrXaDmqUdXFy5Fr8ZvkOpowMjqzMjrT5qCtBSNHZVZWZdqj2low0iqzaqhMZ28PQOSoqfr1yJdaKwXmAkMyRF3HC7PCjp9SbrbohbVVGtTH1/I7qAk6JlYGhdimYSR8+3bgKIxWxYd1AnqkAO1v2j108l+F8uU713EN51klKfV1XNsxMCFhzX5JIQtGgAo/MbEEtO+1a4VeXu3bab+nAKhXggLx9WqjlLadDWrc5IC6oc0t0HPxW44mLoCW1G9c9La8I71cs/w5jB8h1TfumDqn00R/q2rwf87xqt9HFC5znuWZaCW/7N0ppedpuTXvx4pfzU2/kcRvcUybVS1gImBaD5g7REfuBG9cQrcdOiYINxH83Dnx+a+3xO+v5NXQhLf9DgU8Ywy6UAz8168VwwG2drq+Yqd5ATa91A41ojj4GTnNEprIHT1Q1DommtUVQYeeT3O/ZiSqikEM8dx97PeDRRjdQWAV+13eayllb8bqhoNoBrEow2gNZhKo7ye+RxZQ3luTueDaeel6HcDGSXPkTuXpdGs7OxvQxo4UuyrRZAE8WZeLP3fuhPD68HiHGO4y7y3Xq7vtIg23ku47NuGjDzBwfDLoJn2HuxGhS64wvu2unToys3WHmYogFUZbCm4gretGgBfVMmQnfdjTdsehk3NonsSOhEPzJk85X1hjdPrj9PSn2aezi2vtg2moZdYDO3pW7MAbb0/uZOnRAT5/GT9AsbWUl3cHSOv7Y7Xhz/j3KPhu+eKkUKotVvDII3mITJFXqzWdt1OV9dw+jiMSjoLSPidi+D0q37dvj49HwQlMuc6rQu65ZPgj7hV+n4CbY51Nbq7Apb6afJie/1Of6sjNyCB5uiltudNHz7WmvrAifzi01s49iH2LmbeNKIskAjPM4Q9GNbooydd1P3VqEaRKpTvI97CZSZVg7ETXQbjFGhv/uekNg3ag/wdjkjjUjrDLxt6cnyeFlNBfZbcIlEksbUhp3GUHtJWMnOxxMfovjSyNwwVmk9EoLk+8W/aDcVubp787Q7CHyKBefJfIV9yYNp0qEj4nrW+owXrDpQBALBK8kKqT3U0iNrG+l1Dd31at+Y6navOM5sjpMnVgJMBJD6jFu0apbZAcP++rVMpFyA9nF2dXP4JCEQxqXKxR0cImLg9PitEBDT+cjJ/NEQdvl/vuJv8IyOiqoa6lsYKwo1JmV3ruIme1Cg2lSMETAqSglTn40UnVvWKT+zyJAR9TCpIXebk2yRIFtCRabzC7rTIiktYZgnMPcQaW6i1FgVdMmpy6haLC1zLMhErzUF3sRmxACIYpQD+yhgcE+72GPQnceo5VZK8cd0XsugqC+qyiAP1QERk/m3btd9yDWTlvXWNRAGaBEQZgsIuxzbYAJ2iRpCCEFBJYsKw5GbgXIPsgU12D9O3dDQ7vIrBL9KtgmPLyrqaqKjhiTK8Ga3eDSN57/SAHnPa9B4+KklBmxx48d4oasFJlWYvlQwHHilEhEAnUD3/Zb73WL/IHf+41a/CgjScxZqy31QJCJ7xXBAdfqItHajfb7jblC35sn05cs4OingyYX6pLPXVWfQxNRAahOToKPl4zNo9g/+n2W1FVZ0EwmZ1r6T3j4Pnx7Sj4XiGIbjnRLbfOBSnHIsxWxYq9NjnSKZ4AMVxfQ3qmqEyCljXyVB7qaCXB7CEC8g835+ecPDi8fDr99PPsfHo9fc8nF+/59fTqmk9/mZzfTKDJqy0tSEemDqAqI0xp6SLrIINlmjrrAN71g0TmWDYVNjI/nngUUYUSBah3D5EPcqNE1HOCS8xW1b/c4SaO4/WtIHRuBCaeSbBgRkX5+zBddCjR8h+l7/Ympuo8VtNvWTSH7yg3RZekgSftyzDunHTNWS0pDvlufq1+R0CrivLh/ZwsIm24iuHdzwvwIwGMAMClLgEb6RuBf1UJrIdKovN2aM1O3u4PKXDTqtKk7VWWhCJ3sBYUMsOPoTJHISjPGvYhieiTgziBaQpVIvdxdhN4B7y/S/O9A5UN1rcHi7zCau8nfTslyZhROI4WoCSzo2+VdR4L+TqwFpUrHAJmLKCzunKo7xU2eL5DJeaIF2UOFgmNOJY0HlzAT6BNGcM7X3CmsLycbOsDGFPEAcAUEb9jVZYD5wU6/RIOJ8OdyiG+xhCV/IQCbGKMdXmBqXLQFTvWP4upxFNBRfuGN6gywPc7f5OAVQS/reFbNmqvqDB1Qzk7BYP0bQfCvfnOI5gUqwoD7hm98WOhLoHR1+Q8ziPO+87IIIxjHuohvjccqk9rAPHQkmHNhCpuLVAQYS8cxvYQMMr8n46HTfw905tbrZ18HGYclXKoMFGPoasL/RXB+OTgaPQODgz+7uDgco1lb3LnSEwJfTfAjDGlnkGvsGIMrOpJ//BBWvEaUipoJ/Fvjo8/c5iPQ/SbhyZwJxIEHzWRwyTwy5gh3sDD6DBS1/OIkYL8Cu/gWHCVcNIM2uTYe+PVc86xZhYLZm2h7IunvhEAzxT8VHQw8HMbbd+RACiJnob+wYmkX3/xgu74QJUH+r7XFAbsGriZXXBovMaJU4+6ZKa/G3F8rz5pGqF/Er0dB0Z92jk91wMChHtkf+vk+PQ6hY5K9N3gppJ0PauitXuz9/hJSmt9EFw7Zh6Fzf35507/evbGzE2lUvPHyX0SC8eS2QjHmDTNFkhd/kDZxucvPOsXu5zsyX8kXE4yS8cWfxLzKFmkRapWsfTrVioCwea6ad/6PuEnPWjZ6HM9AFeSXJisTpDheIYmSvPxDvhyLI13qORabfMXfxyRLC2dOTqgK+Hd4lq8q9n52TWfXU5nk8vJ9dmnC/7L9PLswxl4oBxMdu2ZzqYX7+Ff72CU691kWAdDgW3jOya7CDX3rq/S9n2341YGt+t/lRIcLgLG0B6BhnLT+nM8W20+AK9rz8S3+G7Xl3D4ov70TaPI4onrin0UyWL+F2TlL7cgmvBsa/CxQckptOIDNOANKW1FoUWSOFQCq/mea+rzx9sdsnprYSzQ1f9fmpuqXRB3NMYz+gMXYsXkiPaGMLPzyQX/dHH+T88CBo6j2iqqM1HdrChdTv/7BiKbZkRDeD5i6mPJVhDSxpSvjUTaMMl1UtuQbr936D+EBeqqXqr6ANf9GDmCA2p+h0tavBUFRV0Z+Jw6iiBNT/AzX3Q7dSLT5MJ+UUFKW23Q8cVpcVdGjDLI+IgJ5AaKBDsSYJ2jaJfY7/5Epy48obpEZYT8hkXS2R+dU+te57I3+sTCR06ZNfIQ2GtK9tc2pfmW5OQ4OO45mQFYWjMboKTjxAVdJ5l4eXPBnhOdaVf7BJsGRFxtxF+79BHbjUbuzibKKooE3Yyaz7/gf2OIawxyVcm4BwZtDpTHqrNq1/11/YuOP/FZoh3ptmdXr81+7ey32Ndw1MipovXUm9HyLegwvBl9LfYOxAwPkKI7hMYo32woAZtT/U6SVUIFku261f33Io7g6rkmoI82l4pCQEbV3jC8U0WaT+x759sYTOJ0VKbXS/AjBIRZzvFu1OMcVYxzbfJUaNX7P9d+r18=')))
print('Extracted', '/kaggle/working/TALLIP_Clean_CPU/src/cpu_train_eval.py')


In [ ]:
# Pin original overlap evidence inside the notebook (non-exhaustive near screen).
evidence_path = ROOT / 'prior_overlap_screen_minimal.zip'
evidence_path.write_bytes(base64.b64decode('UEsDBBQAAAAIAHlkZV1A+MI60wYAANQRAAAlAAAARVZJREVOQ0UvZmFrZW5ld3NfbmVhcl9jYW5kaWRhdGVzLmNzdmWYX26duQ3F3w14J8JAFCmRWk1gTIzWRZAWSdB2b33okmYL8+N37Xupsf2QRPz0jzzn8Ch//O//v15//vry45//+fL2tb2+/Pj29vrjy9v3r6//vf/r57++vf1qP//+9v1v316//OPl999ffnx9fpJubdra2n79eHn73vpvseeIIKK7LY0R7xH5rTO4Vhuq/fH5nnu5Pz8NkbZWHyUSJoOImjTva6y6kGlv4hrzsa/46JYRb2q6H5G9Yu9BxDdzVpmzx7CQ56cp1nSK1Dn8svkc3kboHnVzLtxs67bH5i6+5/MTF2imGX/coq9bZDfn5lY23zGIOAdmxtx1C3dpIrbrQtvHen6KPluIbG//fvl2+zh6tDn9yPQma2RklgVGTOeye0qbIytxP/1Sfm+R4VNLRNbuH3NER0lQN18jAUBIJaSmexlzRHrmbpQbewyxjJg1sdXlwMYg/7aYfY4GVdu+o0DG+E2EiXLvcZS6e+QO1hf5C9mfD0x92ph99ZJabr0JTYC51tJyZBMgk6EhVDaz+rjN0tusSdGtl1mUat9CAT24djm8regZytvrVvOTH6BzGn+UtXQJSARg1sb2yh1zXUm1rdEsC3GPbBtTYVUn/UvVj12GJIbgXcmbbRkZsbyLx6qh7hkaUGGRoTJrGFzPEGwE2rprjeQWWoNSi8/jADpZS7zyZMoeeWLdvblPqp74zuLEBc7BBdvsfdRcDqpPaEKuGJmF9zlz9mtcEtCJhfsUDUsUjEniCa0KkLlyH0jSkJKo5RRNaSG0mlOcRwqo5RwZWtDPuZI8aJnLt0Xaap59SN7SSQBoqtujRUkOPg8uE97vl3lXyBijoYnmH4EVchU5DKkNrSjraFbO2WRgQ+lyYF8phmPDQdAjo1wTncyIAzSDEwXquvPY2lFhCxX7hEHtM5oiop9zrch4kwfb8+RIybwiG1ZV9YgAHLlVcoAeUjrLB7mvkErXIv55jOwVlBxphshFmnMj9kBbpOop7M2iqqIEy1NLCkJVoYBrLylFfjUvmkoQNvVeHwp29akJ/ukK5zogsSG2Y1UV1JVnmvtiTS8YSH5liLOl/Osh6bdRAFB5Pjz1WZ2TWrbeTx1MnVZhdQ+Ra/tIsUNqDmmgYXe+Jh3naLZbVirLZwEyfQEe6WNa+hG5Q9+7rfP4m0WQrziWtp6NCkdQ6uzDM6EQfjafUjxBhPJDSEgohJkVOFwrZw2EWObyQ1tWwtdGZKm716qiXZGugS4iATsfIaFOfoVSXVGr49xApumwQ8I2rMsQhZ3ITukwfVtmyxw9plhnCpA7erVEKRIaPOUKIQY4hc+bwEJ61fxLNgFmildtiTTb1BULcB5jveP2/XMuQSprm7QhCRvbNPuYacg+1KavzCF/Qj6tbQBLIju9EuugG/1wMjgo9tC/4GxCO2Re7S7z9N3MNiSS7ED1SAu25frIPXOqoXhv3DMbN1DWfuyx6BjpIQ83xpaEPL0GexZAj2VpjyaHapCjBLqlWM3U37EOKl9uKUO4qRkQpNbpssHI3ma5WXVMbldFztLkjePQ+ETq6qD0kwdJl9ZozViEewUZw+iE9MM0rVSvNc0KDOaaAzGmm9MTY9UsglXL8xgOo+IZ67dzX66CpZ5HARnkY8jfP41ieKpurp14WrdizNo63iWEEBKSbr7c6+IQraAQHKzPPA4Evl4W1X5Hz7IuTECKUUHVrWWwGgIWUVsx8rKvEPiM2juzoRHwTocE0+e9PfsccK96hQwoqXUB+UOxnYX3OtK0O+YW1uzSfwJNSJOCR4dQR/kYimzoOg8XNq6vAxNwvaA+7NG07HyeTwvH4tYdkJQ8V/Y2aFPfApgqyQiasw67udEEAbc+k/8gVR9F8ckSSH2xoEnAyAuma4IhUhLMO+jK40obxFGqtPZrjxRpnHO1dBOGaYZgIMrRx/kqQo6KnXkf5FTQ4q5UYEEtl/cLt702RYCSLPBED1+dj0oac9BVVn1rrGuYJ0jv9W4OlPJupDE9GZLxyFLKNa1mHf0C38UDMElQB3lScfQNWx+PuQQcYr30/JL34Qzf/RxdKKLNw+7Ae80QkMZieG3ZGPsMDepNraqEg02/Qpf3O6yk80M2I8vOm66iiP6YECY0W/591AfxykQHBqhhDtZn1x52dRheN3cjRZMAlTSq9F6xqwRC/jxfYslG98K+zmMvz0dJWjabqOCU9R7h5Nrrfxb06775/xNZgHmnE5jN1ej9JOKwA3YZ50ggM60XYnBUxmQcYI30rovuU5oCL2lsVOOC1Q1bvswzlK9sSFPqQgNOYdtYSEyNbTvf+Bb5KD0dpORafwJQSwMEFAAAAAgAeWRlXa0i2XnWAAAALwEAACIAAABFVklERU5DRS9mYWtlbmV3c19uZWFyX3NjcmVlbi5qc29uPY9BT8MwDIXv+xVWzxtiUjUm+ANoBziMe+UmLrFIk8p26RDiv5MEiZv93qfn5+8dQDeThey7R+hO/WFkgy2LP2jg9B4Jrjw/owbob8cTjJi8ggVKgM5WjHBB51B8t69JKdtAt4CrGn9SSTRZqTkWhDTkWM/c352bph+8LOQHRVsFrUxGaoPkTSvVGMFtWJClKsfzn4bTRO4fZ9+8h755G0oqveszrnRlX3KhBQAKwcvrG7icJpaZPPh1iewqEXGkqE+gTqj8NuMXzKwKiVDKZi6QdrufX1BLAwQUAAAACAB5ZGVdxwXlRS8CAAAjBQAAJAAAAEVWSURFTkNFL2Zha2VuZXdzX2V4YWN0X292ZXJsYXAuanNvbmVUTW+dMBC851dY75wo+NtOj5V6qZReeqsq5MC+PPQAU2OSVlX/ew3soqS9oNGOd9a7s/j3DWOnAfIltqcHdnr89Pkja8IM59i3LMEQX4BNy9jkJeQujvfzr+Ep9vP9OaYhZPZ66TLMU2iANbHvwzTD6XaVzGG+roLncIURXmeMwpxrCKnvINUplvADU+5/qmtX5luJM8alv92AVHwHWtgdWInAVZqA2wHnSiESHEluKlQQlUBWCCMQGYu5UpoKkT6KG4kq0lWSkKBzTllCBpWll8gq6VFZWaqrnOSEqJoW3B8dYg2tqZqpLGaYcpkDGUTaiwNhDVNScFAVTcMKjdWsFtiRNY5iVqCydXRT6wzWtV54GrMlxKUgpO2BiHUec52n2XvBTQHfN9dfQl/nFLqRtkGaf+JvV0HqwxccqLaKboSUd+R25ThtQCUISeyPe5qIqOi2Qhz7cZwTitN+aJqSMMRKzclt7dEJaUlPcY4xdbBa0apqRffT5kCW+jGKGip+ku/WEWtpo6wS5I4lnxynHXSO9td5hagYQaOqaI+8cuLwZBm7Hwvg+NffcrXFvaU2d3Zi1zpNKU5xhhaz4GfTL3N5L1bvrH5/ZM1+dwA1yjNRD7GFfq6HpTwGT1An2PRK0jnFoX4qD1OdY930oRvKF8K4F+zG56KT0wKbUjdmSFNJ3t6s9RkqlfPdJTashQzNGv3AHr98ZYEVkWt4hrtzAmBrUywmhnVX2Zs/fwFQSwMEFAAAAAgAeWRlXdDi/5t6AQAAZQ4AADUAAABFVklERU5DRS9mYWtlbmV3c19wcm9wb3NlZF9jbGVhbl90cmFpbl9leGNsdXNpb25zLmNzdpWWS24bQQxE9wFyEy3I5q/7NINBNAYECZpgNLBztyxypFzBvfTSb1/VZBeLn/9//53r6345j/X2XI79Y7ldL8e2vvbnzx9v6317bh+vi+Vl+7P+Opf9fTse6+/lfX3crut525/Lfizn9jq/oEMJWrUJwrs3gjePQvgSlL9novcjqhN8Skf6VDA9KyUIvmtD+g8LZ/hhCF+B8CqenRnUNBnBjUWYAYr1gFYwgiXrSk+BhHKmUvhAxtAy5lTtKiilJiLI263VGIhgIowQkiylqRIjdBuQUAPZeyZUaCSZlSOVLCpYStETmc+GGFsjwxwRXMTZ4hEt9Aefc6YxQhoqnDcoq5s31KJz+TN7e6oF3OedpdQb62nvzvrBeybacT7KkJeiKbuSonW2psNDISEDyRrd4SU2BxmqQwwtZbdbqjPCKPSH1MaO85zDlUWw3jokOJI1HZ4/6dlYHcJZT+ecGuzTGZ2lNDsU9UMOZ5UuMZZSzVODRbD81iXwCVBLAwQUAAAACAB5ZGVdaQvXkW8AAABuAQAAMwAAAEVWSURFTkNFL2Zha2VuZXdzX3Byb3Bvc2VkX2NsZWFuX3ZhbF9leGNsdXNpb25zLmNzdoXOMQrDMAxA0b3Qm3hIZCzLpzGiVSA4xMU2Se6WIUfqFbp2CNb+4P/veTWuyWy8xJL3OL9NEa55fT4mTrLKXg0gGjn41WLepCz8iU1q+weehr6wA4194SCAIrwjRZC1fYFolQp6cIqgoHwQQLgVP1BLAwQUAAAACAB5ZGVdgeNExWABAABGBAAAJwAAAEVWSURFTkNFL2hhdGVzcGVlY2hfbmVhcl9jYW5kaWRhdGVzLmNzdm2STU7DQAyF90jcxKrGf2P7NFXURhBUFZRWwN1YcCSugMOiwmmyGCmfbNl+7/18fV/Hy3U/v37spyOMw3yaxnk/nY/j5+3v8naarnB5ns5Pp3H/MhwOw3x8fEASIMrnOg/TGXDXFuaAyNgKjAZCqHcMyf4zEgYUMSzQFLLXo8IcE72OEVdgtlagZSNiRNnSomel3irbLjS//vjgsqwaXFZtzkCOfZvaFpUqAKIDN9KoWgUBh2G/o6IYCu/DqVbSZj/TJtUV5b9ZK7dSl6RuK6oblInAGhZzsIvnuU3rCpZes+BqMScFFS0bUEtVkZ0rJcbUy6q2tMTq7gaSTBI5V3dIM5fM3bTS9CUTQqvaHuBec5NeCQRHhXk/CGsedvPmj1HXqjcZEpBZ90ozeH3J6T/IKRd4BG1BuYPRem3vmAqmByUbHGpAzWviJEMOjBq6SYu10lMQTnXLBWKetLea719QSwMEFAAAAAgAeWRlXbc/zYHWAAAALgEAACQAAABFVklERU5DRS9oYXRlc3BlZWNoX25lYXJfc2NyZWVuLmpzb249T0FOxDAMvO8rrJ5ZBNqyrOADiMNygHvlJi6xNk0q26WLEH8nyYGbPTMez/zsALqZLGTfPUF37PcjG2xZ/F4Dp89I8M7zC2qA/np/hBGTV7BACdDZihFe0TkU391Up5RtoGvAVY2/qDiarNQYC0Iacqxv7m5PDw3UCy8L+UHRVkErk5HaIHnTKmsawW1YkKUi/alBOE3k/tXsK3V4bNSGkkrs2sWVqOyLK7RzQCE4v32Ay2limcmDX5fIrioijhT1GdQJlWozfsPMqpAIpWzmAmm3+/0DUEsDBBQAAAAIAHlkZV0HsuqLzwEAAAEEAAAmAAAARVZJREVOQ0UvaGF0ZXNwZWVjaF9leGFjdF9vdmVybGFwLmpzb25lU8tu3SAQ3ecr0F0nCi8b3C4rZVMp3XRXVYhrz42t2MYBnNuq6r8XbMZK0g06msc5Mwf4c0PIaYLYu+70iZweH75+Ia0NcHFjRzxM7hXIss5tXG0c3Hwffk9nN4b7i/OTjeTaDxHCYlsgrRtHuwQ43WbKaMNzJuxtzgO0fYlDiAasHwfwxrtrSEWC/58aupz5keKEMC5vC9AFNHQHXIoCVIWg1EhdIko3O9CydDGqSxtjDCl5c6g0pZMJzguqJdYphlnNUZPWOAYXDCc7Zqxwfl7JI1Y3iFCXK4q9inFETb0jkYQLqllRE02lyrI1LcxSaZrAz83SVzua6O0wo9VcfIi/9VnJoqApWiFVkWcVQ/OUwJjSNRqAN8OpwnWYwhVljescl8Irih1aSFwRWYSg6kAYk5J/NEBSvAzJcHgpmToQvoVGscOUdR5eVij750eXnyB7m9rs2RN7fPFucQG60gS/2nEN6Ttk7+rqfUluflewU8xwNZPrYAxmWtNDP4PxsNGlnot3kzmnb2eiM+1ohymdYOddb5ifEk30K2xMwxzBL6l5+5H5kyXheNe7lnQQoc3Rz+Tx23diSSJ5tk9wd/EAJK9EnCdFN9Pe/P0HUEsDBBQAAAAIAHlkZV1RqR0JNwEAAP0MAAA3AAAARVZJREVOQ0UvaGF0ZXNwZWVjaF9wcm9wb3NlZF9jbGVhbl90cmFpbl9leGNsdXNpb25zLmNzdp3XTW6DMBCG4X2l3oSF58ee8WkQSpCCGoWIoLR366JH6hXKtru+3fMIfzP2GL4/v/bp8Tbs27Tcxm19H5fzsM3TY729vlymfX7c5/l0GcQjh/ljOu3j+py363Qfn9N1OU/7st7GdRuPJ/dfQlWdCRcNKkKhUKlUdGOieoXJa2SDIhsUzRMmjyLCRNZULGB1M0uhotNV9X8I2PNeGuxHtw6Ti5hggpOIZumQWIpjYpS4OSYecNNLVqNF7k04wfF70oVpKaqUZIXdP2ZwUqKNji/13mDFtFXtmOAiH0Q4UU6ME1rkcNz9iEa/JbJUGj+lGSdBifG3NDotrWjFJGgWkwzHJOHmt+NYwu6bCb0szFpUSoJ+WJgLPZVW1RITp1lacZolisKdbD2kYUInzEGS/hYcl9hfsvwAUEsDBBQAAAAIAHlkZV1oHn+vQAAAAD8AAAA1AAAARVZJREVOQ0UvaGF0ZXNwZWVjaF9wcm9wb3NlZF9jbGVhbl92YWxfZXhjbHVzaW9ucy5jc3Z7v3t/SWJxtk5ZYk58UX55fGaKTlFqYnF+Hi9XRmJJanFBampyho6xsYGFTmpFYnJJfH5ZalFOYkE8UK6ElwsAUEsBAhQDFAAAAAgAeWRlXUD4wjrTBgAA1BEAACUAAAAAAAAAAAAAAIABAAAAAEVWSURFTkNFL2Zha2VuZXdzX25lYXJfY2FuZGlkYXRlcy5jc3ZQSwECFAMUAAAACAB5ZGVdrSLZedYAAAAvAQAAIgAAAAAAAAAAAAAAgAEWBwAARVZJREVOQ0UvZmFrZW5ld3NfbmVhcl9zY3JlZW4uanNvblBLAQIUAxQAAAAIAHlkZV3HBeVFLwIAACMFAAAkAAAAAAAAAAAAAACAASwIAABFVklERU5DRS9mYWtlbmV3c19leGFjdF9vdmVybGFwLmpzb25QSwECFAMUAAAACAB5ZGVd0OL/m3oBAABlDgAANQAAAAAAAAAAAAAAgAGdCgAARVZJREVOQ0UvZmFrZW5ld3NfcHJvcG9zZWRfY2xlYW5fdHJhaW5fZXhjbHVzaW9ucy5jc3ZQSwECFAMUAAAACAB5ZGVdaQvXkW8AAABuAQAAMwAAAAAAAAAAAAAAgAFqDAAARVZJREVOQ0UvZmFrZW5ld3NfcHJvcG9zZWRfY2xlYW5fdmFsX2V4Y2x1c2lvbnMuY3N2UEsBAhQDFAAAAAgAeWRlXYHjRMVgAQAARgQAACcAAAAAAAAAAAAAAIABKg0AAEVWSURFTkNFL2hhdGVzcGVlY2hfbmVhcl9jYW5kaWRhdGVzLmNzdlBLAQIUAxQAAAAIAHlkZV23P82B1gAAAC4BAAAkAAAAAAAAAAAAAACAAc8OAABFVklERU5DRS9oYXRlc3BlZWNoX25lYXJfc2NyZWVuLmpzb25QSwECFAMUAAAACAB5ZGVdB7Lqi88BAAABBAAAJgAAAAAAAAAAAAAAgAHnDwAARVZJREVOQ0UvaGF0ZXNwZWVjaF9leGFjdF9vdmVybGFwLmpzb25QSwECFAMUAAAACAB5ZGVdUakdCTcBAAD9DAAANwAAAAAAAAAAAAAAgAH6EQAARVZJREVOQ0UvaGF0ZXNwZWVjaF9wcm9wb3NlZF9jbGVhbl90cmFpbl9leGNsdXNpb25zLmNzdlBLAQIUAxQAAAAIAHlkZV1oHn+vQAAAAD8AAAA1AAAAAAAAAAAAAACAAYYTAABFVklERU5DRS9oYXRlc3BlZWNoX3Byb3Bvc2VkX2NsZWFuX3ZhbF9leGNsdXNpb25zLmNzdlBLBQYAAAAACgAKAHwDAAAZFAAAAAA='))
print('Prior evidence:', evidence_path, evidence_path.stat().st_size, 'bytes')


In [ ]:

# Resolve the two attached input datasets; verify expected schema before use.
def find_zip(kind, explicit):
    if explicit:
        target = Path(explicit)
        assert target.is_file(), f'ZIP not found: {target}'
        return target
    zips = sorted(INPUT_ROOT.rglob('*.zip'))
    if kind == 'fakenews':
        matches = [p for p in zips if 'multi-source' in p.name.lower() and 'fake' in p.name.lower()]
    else:
        matches = [p for p in zips if 'bd-shs' in p.name.lower() or 'bdshs' in p.name.lower()]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise RuntimeError(f'Ambiguous {kind} ZIPs: {matches}; set the explicit ZIP path')

    matches = []
    for train in INPUT_ROOT.rglob('train.csv'):
        folder = train.parent
        if not (folder / 'val.csv').exists() or not (folder / 'test.csv').exists():
            continue
        header = pd.read_csv(train, nrows=1).columns
        is_kind = ('text' in header and 'label' in header) if kind == 'fakenews' else ('sentence' in header and 'hate speech' in header)
        if is_kind:
            matches.append(folder)
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {kind} CSV directory, found: {matches}; specify path manually')
    folder = matches[0]
    archive = ROOT / (kind + '_original_input.zip')
    if not archive.exists():
        with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_STORED) as z:
            for split in ('train', 'val', 'test'):
                arc = ('data/processed/' if kind == 'fakenews' else '') + split + '.csv'
                z.write(folder / (split + '.csv'), arc)
    return archive

fake_zip = find_zip('fakenews', FAKENEWS_ZIP)
hate_zip = find_zip('hatespeech', HATESPEECH_ZIP)
print('Fake News source:', fake_zip)
print('Hate Speech source:', hate_zip)


In [ ]:
# (Re)build clean splits from the original CSVs, independently verify overlap evidence.
subprocess.run([sys.executable, str(ROOT/'src'/'clean_splits.py'),
    '--fakenews-zip', str(fake_zip), '--hatespeech-zip', str(hate_zip),
    '--evidence-zip', str(evidence_path), '--out-dir', str(ROOT)], check=True)
manifest_path = ROOT/'evidence'/'clean_split_manifest.json'
manifest = json.loads(manifest_path.read_text())
for task, details in manifest['tasks'].items():
    print(task, 'original', details['original_rows'], 'CLEAN', details['clean_rows'],
          'exact overlap after cleaning:', details['post_clean_exact_cross_split_overlap'])
# Remove only locally repacked duplicate ZIPs; originals remain in Kaggle Input.
for dataset_zip in (fake_zip, hate_zip):
    if dataset_zip.parent == ROOT and dataset_zip.name.endswith('_original_input.zip'):
        dataset_zip.unlink(missing_ok=True)
        print('Removed redundant working copy:', dataset_zip.name)


In [ ]:
# Optionally restore the results of this *same cleaned-split* experiment.
training_dir = ROOT/'training'
old_manifest = None
if PREVIOUS_RESULTS_DIR:
    old = Path(PREVIOUS_RESULTS_DIR)
    assert old.is_dir(), f'Previous training folder missing: {old}'
    assert (old/'run_plan.json').exists() or (old/'cpu_run_plan.json').exists(), 'Not a prior TALLIP clean-training folder'
    shutil.copytree(old, training_dir, dirs_exist_ok=True)
    candidate = old.parent/'evidence'/'clean_split_manifest.json'
    if candidate.exists():
        old_manifest = candidate

plan_cmd = [sys.executable, str(ROOT/'src'/'cpu_train_eval.py'),
  '--clean-dir', str(ROOT/'cleaned_data'),
  '--manifest', str(ROOT/'evidence'/'clean_split_manifest.json'),
  '--out-dir', str(training_dir),
  '--micro-batch', str(CPU_MICRO_BATCH), '--eval-batch', str(CPU_EVAL_BATCH),
  '--threads', str(CPU_THREADS),
  '--checkpoint-every', str(CPU_CHECKPOINT_EVERY_UPDATES),
  '--max-wall-hours', str(CPU_MAX_WALL_HOURS),
  '--runs', *REMAINING_NINE]
if old_manifest:
    plan_cmd += ['--previous-manifest', str(old_manifest)]
subprocess.run(plan_cmd + ['--plan-only'], check=True)
print('CPU plan checks passed: exactly NINE remaining runs; existing three are NOT trained again.')


In [ ]:
# Actual CPU full-fine-tuning. No GPU is used or required.
# Stops automatically at the wall-time limit *after saving* a checkpoint.
# Multiple sessions are normally necessary. Do not expect nine runs in a day.
subprocess.run(plan_cmd, check=True)
print('CPU session finished or paused safely. Save Kaggle output before the next session.')


In [ ]:
# Report only legitimately completed runs.
from pathlib import Path
done_files = sorted((training_dir).glob('**/done.json'))
print('COMPLETE TRAINING RUNS PRESENT:', len(done_files))
for path in done_files:
    data = json.loads(path.read_text())
    print(path.relative_to(training_dir), data.get('status'),
          'test macro F1:', data.get('fixed_original_test',{}).get('macro_f1'))
print('Checkpointed/unfinished runs:')
for path in sorted(training_dir.glob('**/latest_epoch.pt')):
    print(path.relative_to(training_dir))
print('OUTPUT ROOT:', ROOT)
